# TASTF: Tier-Aware Spatiotemporal Forecasting for HetNets

This notebook implements the **TASTF** (Tier-Aware Spatiotemporal Forecasting) model for Heterogeneous Cellular Networks (HetNets). 

### Core Contribution
TASTF uses a **Heterogeneous Graph Convolutional Network** that treats Macro, Pico, and Femto base stations as distinct node types with type-specific convolutional layers via PyTorch Geometric `HeteroConv`. This approach explicitly models the hierarchy and cross-tier influence in HetNets, which is then processed by a **Transformer Encoder** to capture long-range temporal dependencies.

---

## 1. Environment Setup & Dependencies

In [ ]:
# Pre-built wheels (data.pyg.org) — avoids 20+ min source builds for torch-scatter/torch-sparse.
import subprocess
import sys
import torch

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip_install("pandas", "numpy", "scikit-learn", "matplotlib")

t = torch.__version__.split("+")[0]
cuda = ("cu" + torch.version.cuda.replace(".", "")) if torch.version.cuda else "cpu"
index_url = f"https://data.pyg.org/whl/torch-{t}+{cuda}.html"
print(f"Installing PyG extensions from: torch-{t}+{cuda}")

try:
    pip_install("torch-scatter", "torch-sparse", "-f", index_url)
except subprocess.CalledProcessError:
    # If your exact torch patch isn't published, try X.Y.0 wheels (same major.minor).
    parts = t.split(".")
    if len(parts) >= 3:
        t_alt = f"{parts[0]}.{parts[1]}.0"
        index_url = f"https://data.pyg.org/whl/torch-{t_alt}+{cuda}.html"
        print(f"Retrying with: torch-{t_alt}+{cuda}")
        pip_install("torch-scatter", "torch-sparse", "-f", index_url)
    else:
        raise

pip_install("torch-geometric")
print("Done.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.data import HeteroData

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

## 2. Training (canonical pipeline)

Runs the same code as 
eference_implementation/train.py: baselines, chronological split, temporal features, inverse metrics.  
**Data:** place Milan *.txt files in hetnet-traffic-forecast/Wireless Dataset/ (or set DATA_DIR below).


In [ ]:
import os
import sys
from pathlib import Path

# Repo root = parent of this notebook (when notebook lives in hetnet-traffic-forecast/)
NB = Path.cwd().resolve()
for root in (NB, NB.parent, NB / 'hetnet-traffic-forecast'):
    ref = root / 'reference_implementation'
    if ref.is_dir():
        sys.path.insert(0, str(ref))
        REPO = root
        break
else:
    raise FileNotFoundError('reference_implementation/ not found; open notebook from repo folder.')

from train import run_training
from paths import resolve_wireless_dataset_dir

# Override if needed (Colab: upload Wireless Dataset and set path here)
DATA_DIR = os.environ.get('TASTF_DATA_DIR') or resolve_wireless_dataset_dir(None)
print('DATA_DIR:', DATA_DIR)

run_training(
    data_dir=DATA_DIR,
    epochs=50,
    batch_size=32,
    seed=42,
)


## 3. Metrics & figures

Requires 
eference_implementation/results.npz from the training cell.


In [ ]:
import os, sys
from pathlib import Path
NB = Path.cwd().resolve()
for root in (NB, NB.parent):
    ref = root / 'reference_implementation'
    if ref.is_dir():
        sys.path.insert(0, str(ref))
        os.chdir(ref)
        break

import read_metrics
read_metrics.main('results.npz')

from evaluate import plot_tier_predictions
plot_tier_predictions('results.npz', 'tier_predictions.png')

from viz import plot_error_map_from_npz
plot_error_map_from_npz('results.npz', 'spatial_error_map.png')
